# Fama and French Five Factor Model (FFFFM)

## What is FFFFM?
The Fama and French Five Factor Model is a **multiple linear regression**
model used to predict stock returns. It was developed by Eugene Fama and
Kenneth French (2015) as an extension of their earlier three-factor model.

It is considered the **"best of regression"** algorithms for financial
stock return prediction among linear models.

## The Five Factor Equation

    R_it - R_Ft = a_i + b_i(R_Mt - R_Ft) + s_i*SMB + h_i*HML + r_i*RMW + c_i*CMA + e_it

Where:
- R_it       → return on stock i at time t
- R_Ft       → risk-free return (T-bill rate)
- R_it-R_Ft  → excess return (this is our TARGET variable y)
- R_Mt-R_Ft  → excess market return        (Factor 1)
- SMB        → Small Minus Big             (Factor 2) - size factor
- HML        → High Minus Low              (Factor 3) - value factor
- RMW        → Robust Minus Weak           (Factor 4) - profitability factor
- CMA        → Conservative Minus Aggressive (Factor 5) - investment factor
- a_i        → intercept (should be ~0 if model is perfect)
- e_it       → error term

## What Each Factor Means
- **Mkt-RF (Market Premium)**: Extra return of market over risk-free rate
- **SMB (Size)**: Small cap stocks tend to outperform large cap stocks
- **HML (Value)**: High book-to-market stocks outperform low book-to-market
- **RMW (Profitability)**: Profitable firms outperform unprofitable firms
- **CMA (Investment)**: Conservative investing firms outperform aggressive ones

## Why Use It?
- Acts as our **baseline linear model**
- If FFFFM performs well, the five factors are strong predictors
- We compare GRU and SVR against this to see if nonlinear models improve prediction

## Evaluation Metric — R²
R² (coefficient of determination) measures how well the model explains
the variance in stock returns:
- R² = 1 → perfect prediction
- R² = 0 → model explains nothing
- Higher R² = better model

In [2]:
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import cross_val_score, KFold
from scipy import stats

# ── ENVIRONMENT SETUP ──────────────────────────────────────
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/portfolio_project'
else:
    BASE_DIR = os.path.join(os.getcwd(), '..','data','portfolio_project')

print(f"Using: {BASE_DIR}")

# ── LOAD DATA ──────────────────────────────────────────────
train = pd.read_csv(os.path.join(BASE_DIR, 'train_data.csv'),
                    index_col='Date', parse_dates=True)
valid = pd.read_csv(os.path.join(BASE_DIR, 'valid_data.csv'),
                    index_col='Date', parse_dates=True)

print(f"Train: {train.shape}")
print(f"Valid: {valid.shape}")

# ── FEATURES & TARGETS ─────────────────────────────────────
feature_cols = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']

X_train = train[feature_cols].values
X_valid = valid[feature_cols].values

targets = {
    'MSFT': (train['MSFT_excess'].values, valid['MSFT_excess'].values),
    'AAPL': (train['AAPL_excess'].values, valid['AAPL_excess'].values),
    'SONY': (train['SONY_excess'].values, valid['SONY_excess'].values),
}

Using: /Users/aaronjasonbaptist/Documents/IIT Kharagpur/Academic/Semester 4/Deep Learning /Github/gru-portfolio-management-project-main/models/../data/portfolio_project
Train: (2999, 12)
Valid: (499, 12)


## Training the FFFFM

- We train **3 separate regression models** — one per stock
- Each model uses the same 5 factors as input (X)
- But predicts the excess return of its own stock (y)
- Training data: Jan 2000 – Dec 2011 (~2996 days)
- Validation data: Dec 2011 – Nov 2013 (~504 days)
- We use **Ordinary Least Squares (OLS)** to fit the model

In [3]:
ffffm_models     = {}
ffffm_r2         = {}
ffffm_preds      = {}

print("=" * 50)
print("FFFFM — Training & Validation Results")
print("=" * 50)

for stock, (y_train, y_valid) in targets.items():

    # TRAIN
    model = LinearRegression()
    model.fit(X_train, y_train)

    # PREDICT
    y_pred = model.predict(X_valid)

    # R² SCORE
    r2 = r2_score(y_valid, y_pred)

    # Store
    ffffm_models[stock] = model
    ffffm_r2[stock]     = r2
    ffffm_preds[stock]  = y_pred

    # Print coefficients
    print(f"\n{stock}:")
    print(f"  Intercept : {model.intercept_:.6f}")
    for fname, coef in zip(feature_cols, model.coef_):
        print(f"  {fname:8s} : {coef:.6f}")
    print(f"  R²        : {r2:.4f}")

print("\n" + "=" * 50)
print("Summary R² values:")
for stock, r2 in ffffm_r2.items():
    print(f"  {stock}: {r2:.3f}")

FFFFM — Training & Validation Results

MSFT:
  Intercept : 0.000172
  Mkt-RF   : 1.028760
  SMB      : -0.260792
  HML      : -0.396946
  RMW      : -0.159382
  CMA      : -0.253083
  R²        : 0.2852

AAPL:
  Intercept : 0.001643
  Mkt-RF   : 1.025547
  SMB      : 0.014179
  HML      : -0.401451
  RMW      : -0.369842
  CMA      : -0.876220
  R²        : 0.2248

SONY:
  Intercept : -0.000420
  Mkt-RF   : 0.985401
  SMB      : -0.065129
  HML      : -0.142071
  RMW      : -0.160267
  CMA      : 0.070251
  R²        : 0.2078

Summary R² values:
  MSFT: 0.285
  AAPL: 0.225
  SONY: 0.208


## 20-Fold Cross Validation & P-value

To verify the **consistency and stability** of our results, we use
20-fold cross validation:

- The training data is split into 20 equal parts
- Model trains on 19 parts and validates on 1 part
- This repeats 20 times
- We collect 20 R² scores and compute mean and std deviation

**P-value test:**
- Null hypothesis H₀: The mean R² from cross validation
  is NOT equal to the R² we reported
- If p-value < 0.05 → reject H₀ → our results are
  **consistent and statistically significant**
- Paper reports p < 0.05 for all three models

In [4]:
print("=" * 50)
print("FFFFM — 20-Fold Cross Validation & P-values")
print("=" * 50)

# Combine train+valid for cross validation
X_all = np.vstack([X_train, X_valid])

kf = KFold(n_splits=20, shuffle=False)

for stock, (y_train, y_valid) in targets.items():
    y_all = np.concatenate([y_train, y_valid])

    model = LinearRegression()

    # 20 R² scores
    cv_scores = cross_val_score(model, X_all, y_all,
                                cv=kf, scoring='r2')

    mean_r2 = cv_scores.mean()
    std_r2  = cv_scores.std()

    # P-value: test if mean cv R² equals reported R²
    t_stat, p_value = stats.ttest_1samp(cv_scores, ffffm_r2[stock])

    print(f"\n{stock}:")
    print(f"  Reported R²     : {ffffm_r2[stock]:.3f}")
    print(f"  CV Mean R²      : {mean_r2:.3f}")
    print(f"  CV Std R²       : {std_r2:.3f}")
    print(f"  P-value         : {p_value:.4f}")
    print(f"  Significant?    : {'Yes' if p_value < 0.05 else 'No'}")

FFFFM — 20-Fold Cross Validation & P-values

MSFT:
  Reported R²     : 0.285
  CV Mean R²      : 0.439
  CV Std R²       : 0.158
  P-value         : 0.0004
  Significant?    : Yes

AAPL:
  Reported R²     : 0.225
  CV Mean R²      : 0.316
  CV Std R²       : 0.138
  P-value         : 0.0096
  Significant?    : Yes

SONY:
  Reported R²     : 0.208
  CV Mean R²      : 0.296
  CV Std R²       : 0.100
  P-value         : 0.0011
  Significant?    : Yes


In [5]:
# Save FFFFM predictions on validation set
ffffm_preds_df = pd.DataFrame(ffffm_preds, index=valid.index)
ffffm_preds_df.to_csv(os.path.join(BASE_DIR, 'ffffm_predictions.csv'))
print("Saved FFFFM predictions!")
print(ffffm_preds_df.head())

Saved FFFFM predictions!
                MSFT      AAPL      SONY
Date                                    
2011-12-05  0.011870  0.018209  0.010640
2011-12-06  0.000002 -0.000241  0.000089
2011-12-07 -0.001712 -0.003586  0.000465
2011-12-08 -0.018232 -0.020830 -0.021992
2011-12-09  0.016083  0.023184  0.016928
